# CEHARPS 01 — ดาวน์โหลดและตรวจชุดข้อมูล / Download and validate datasets

ส่วนนี้ดาวน์โหลด MagSet-2 public sample จำนวน 23 คู่ภาพ–หน้ากากเพื่อทดสอบ U-Net และ DeepLabV3+ พร้อมตรวจ SHA-256 ก่อนแตกไฟล์

MagSet-2 มีป้ายถิ่นที่อยู่ป่าชายเลน แต่ไม่มีป้ายสุขภาพ MHI จึงสร้างข้อมูลสุขภาพจำลองแยกต่างหากเมื่อ `health_mode="demo"` ข้อมูลจำลองห้ามใช้รายงานผลการแข่งขัน


In [ ]:
import hashlib  # TH: นำเข้าเครื่องมือตรวจลายนิ้วมือไฟล์ | EN: Import file-fingerprint utilities.
import json  # TH: นำเข้าเครื่องมือ JSON | EN: Import JSON utilities.
import shutil  # TH: นำเข้าเครื่องมือคัดลอกไฟล์ | EN: Import file-copying utilities.
import urllib.request  # TH: นำเข้าเครื่องมือดาวน์โหลดผ่าน URL | EN: Import URL download utilities.
import zipfile  # TH: นำเข้าเครื่องมือจัดการ ZIP | EN: Import ZIP archive utilities.
from pathlib import Path  # TH: นำเข้าคลาสจัดการพาธ | EN: Import the path-management class.
import numpy as np  # TH: นำเข้า NumPy สำหรับสร้างข้อมูลตัวเลข | EN: Import NumPy for numerical data.
import pandas as pd  # TH: นำเข้า pandas สำหรับตารางข้อมูล | EN: Import pandas for tabular data.
from google.colab import drive  # TH: นำเข้าเครื่องมือเชื่อม Drive | EN: Import the Drive connector.
drive.mount("/content/drive")  # TH: เชื่อม Google Drive | EN: Mount Google Drive.
PROJECT_ROOT = Path("/content/drive/MyDrive/CEHARPS")  # TH: กำหนดโฟลเดอร์โครงการ | EN: Define the project folder.
CONFIG = json.loads((PROJECT_ROOT / "config.json").read_text(encoding="utf-8"))  # TH: อ่านค่ากลางจากส่วน 00 | EN: Load settings created in part 00.
SEED = int(CONFIG["seed"])  # TH: อ่านค่าเมล็ดสุ่ม | EN: Read the random seed.


In [ ]:
SAMPLE_URL = "https://raw.githubusercontent.com/lucasjvds/MangroveAI/main/src/dataset.zip"  # TH: กำหนด URL ของ MagSet-2 sample | EN: Define the MagSet-2 sample URL.
SAMPLE_SHA256 = "c2fcb1be182e1cbb40cbecd835e6c32a51a256c8ea7045904d968aa06e2a3d1c"  # TH: กำหนด SHA-256 ที่คาดหวัง | EN: Define the expected SHA-256 digest.
ARCHIVE = PROJECT_ROOT / "data/raw/magset2_dataset.zip"  # TH: กำหนดตำแหน่งไฟล์ ZIP | EN: Define the ZIP destination.
EXTRACTED = PROJECT_ROOT / "data/raw/magset2_extracted"  # TH: กำหนดโฟลเดอร์แตกไฟล์ | EN: Define the extraction folder.
SEGMENTATION_ROOT = PROJECT_ROOT / "data/segmentation/magset2_sample"  # TH: กำหนดโฟลเดอร์ข้อมูลภาพพร้อมใช้ | EN: Define the prepared image-data folder.

def sha256_file(path: Path) -> str:  # TH: สร้างฟังก์ชันคำนวณ SHA-256 | EN: Define a SHA-256 calculation function.
    digest = hashlib.sha256()  # TH: สร้างตัวสะสมค่าแฮช | EN: Create a hash accumulator.
    with path.open("rb") as stream:  # TH: เปิดไฟล์แบบไบนารี | EN: Open the file in binary mode.
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):  # TH: อ่านไฟล์ทีละ 1 MB | EN: Read the file in 1 MB chunks.
            digest.update(chunk)  # TH: ป้อนข้อมูลส่วนนี้เข้าแฮช | EN: Add the chunk to the digest.
    return digest.hexdigest()  # TH: คืนค่าแฮชเป็นข้อความ | EN: Return the hexadecimal digest.

def download_checked() -> None:  # TH: สร้างฟังก์ชันดาวน์โหลดพร้อมตรวจไฟล์ | EN: Define a verified-download function.
    ARCHIVE.parent.mkdir(parents=True, exist_ok=True)  # TH: สร้างโฟลเดอร์ปลายทาง | EN: Create the destination folder.
    if ARCHIVE.exists() and sha256_file(ARCHIVE) == SAMPLE_SHA256:  # TH: ตรวจว่าไฟล์เดิมถูกต้องหรือไม่ | EN: Check whether an existing file is valid.
        print("ใช้ไฟล์เดิมที่ผ่าน checksum / Reusing verified archive")  # TH: แจ้งว่าใช้ไฟล์เดิม | EN: Report reuse of the verified archive.
        return  # TH: จบฟังก์ชันโดยไม่ดาวน์โหลดซ้ำ | EN: Exit without downloading again.
    temporary = ARCHIVE.with_suffix(".zip.part")  # TH: กำหนดชื่อไฟล์ชั่วคราว | EN: Define a temporary download path.
    urllib.request.urlretrieve(SAMPLE_URL, temporary)  # TH: ดาวน์โหลดข้อมูลจากอินเทอร์เน็ต | EN: Download the dataset from the internet.
    actual = sha256_file(temporary)  # TH: คำนวณแฮชไฟล์ที่ดาวน์โหลด | EN: Hash the downloaded file.
    if actual != SAMPLE_SHA256:  # TH: ตรวจว่าค่าแฮชตรงหรือไม่ | EN: Check whether the digest matches.
        temporary.unlink(missing_ok=True)  # TH: ลบไฟล์ที่ไม่ผ่านการตรวจ | EN: Delete the failed download.
        raise ValueError(f"Checksum mismatch: {actual}")  # TH: หยุดเมื่อไฟล์ไม่ถูกต้อง | EN: Stop when file integrity fails.
    temporary.replace(ARCHIVE)  # TH: เปลี่ยนไฟล์ชั่วคราวเป็นไฟล์จริง | EN: Promote the temporary file to the final archive.

def safe_extract() -> None:  # TH: สร้างฟังก์ชันแตก ZIP อย่างปลอดภัย | EN: Define a safe ZIP extraction function.
    EXTRACTED.mkdir(parents=True, exist_ok=True)  # TH: สร้างโฟลเดอร์แตกไฟล์ | EN: Create the extraction folder.
    root = EXTRACTED.resolve()  # TH: แปลงพาธรากเป็นพาธสมบูรณ์ | EN: Resolve the extraction root.
    with zipfile.ZipFile(ARCHIVE) as archive:  # TH: เปิดไฟล์ ZIP | EN: Open the ZIP archive.
        for member in archive.infolist():  # TH: วนตรวจสมาชิกทุกไฟล์ | EN: Inspect every archive member.
            target = (EXTRACTED / member.filename).resolve()  # TH: คำนวณพาธปลายทาง | EN: Resolve the member destination.
            if root != target and root not in target.parents:  # TH: ป้องกัน path traversal | EN: Prevent path traversal.
                raise ValueError(f"Unsafe ZIP member: {member.filename}")  # TH: หยุดเมื่อพบพาธอันตราย | EN: Stop on an unsafe archive path.
        archive.extractall(EXTRACTED)  # TH: แตกไฟล์หลังตรวจครบ | EN: Extract after validation succeeds.

download_checked()  # TH: ดาวน์โหลดและตรวจไฟล์ | EN: Download and verify the archive.
safe_extract()  # TH: แตกไฟล์อย่างปลอดภัย | EN: Safely extract the archive.


In [ ]:
SOURCE_DATASET = EXTRACTED / "dataset"  # TH: กำหนดโฟลเดอร์ข้อมูลต้นฉบับ | EN: Define the extracted source folder.
image_by_stem = {path.stem: path for path in (SOURCE_DATASET / "satellite-images").glob("*.tiff")}  # TH: สร้างดัชนีภาพตามชื่อ | EN: Index images by filename stem.
mask_by_stem = {path.stem: path for path in (SOURCE_DATASET / "masks").glob("*.npy")}  # TH: สร้างดัชนีหน้ากากตามชื่อ | EN: Index masks by filename stem.
stems = sorted(set(image_by_stem) & set(mask_by_stem))  # TH: เลือกชื่อที่มีทั้งภาพและหน้ากาก | EN: Keep stems with both image and mask.
if len(stems) != 23:  # TH: ตรวจจำนวนคู่ข้อมูลที่คาดหวัง | EN: Validate the expected pair count.
    raise ValueError(f"Expected 23 image-mask pairs, found {len(stems)}")  # TH: หยุดเมื่อจำนวนผิด | EN: Stop when the count is wrong.
rng = np.random.default_rng(SEED)  # TH: สร้างตัวสุ่มที่ทำซ้ำได้ | EN: Create a reproducible random generator.
shuffled = np.array(stems, dtype=object)  # TH: แปลงรายชื่อเป็นอาร์เรย์ | EN: Convert stems to an array.
rng.shuffle(shuffled)  # TH: สับลำดับข้อมูล | EN: Shuffle the sample order.
split_map = {str(stem): ("train" if index < 16 else "val" if index < 19 else "test") for index, stem in enumerate(shuffled)}  # TH: แบ่ง 16/3/4 สำหรับสาธิต | EN: Create a 16/3/4 demonstration split.
records = []  # TH: เตรียมรายการเมทาดาทา | EN: Initialize metadata records.
for stem in stems:  # TH: วนจัดเตรียมทุกคู่ข้อมูล | EN: Prepare every image-mask pair.
    split = split_map[stem]  # TH: อ่านชุดแบ่งของตัวอย่าง | EN: Read the sample split.
    image_dir = SEGMENTATION_ROOT / "images" / split  # TH: กำหนดโฟลเดอร์ภาพปลายทาง | EN: Define the image destination.
    mask_dir = SEGMENTATION_ROOT / "masks" / split  # TH: กำหนดโฟลเดอร์หน้ากากปลายทาง | EN: Define the mask destination.
    image_dir.mkdir(parents=True, exist_ok=True)  # TH: สร้างโฟลเดอร์ภาพ | EN: Create the image folder.
    mask_dir.mkdir(parents=True, exist_ok=True)  # TH: สร้างโฟลเดอร์หน้ากาก | EN: Create the mask folder.
    image_output = image_dir / image_by_stem[stem].name  # TH: กำหนดไฟล์ภาพปลายทาง | EN: Define the output image path.
    mask_output = mask_dir / mask_by_stem[stem].name  # TH: กำหนดไฟล์หน้ากากปลายทาง | EN: Define the output mask path.
    shutil.copy2(image_by_stem[stem], image_output)  # TH: คัดลอกภาพพร้อมเมทาดาทา | EN: Copy the image with metadata.
    shutil.copy2(mask_by_stem[stem], mask_output)  # TH: คัดลอกหน้ากากพร้อมเมทาดาทา | EN: Copy the mask with metadata.
    records.append({"sample_id": stem, "split": split, "image": str(image_output), "mask": str(mask_output), "label_type": "habitat", "data_status": "public_sample"})  # TH: บันทึกหลักฐานกำกับตัวอย่าง | EN: Record sample provenance.
manifest = pd.DataFrame(records).sort_values(["split", "sample_id"])  # TH: สร้างและเรียงตาราง manifest | EN: Build and sort the manifest table.
manifest.to_csv(SEGMENTATION_ROOT / "manifest.csv", index=False)  # TH: บันทึก manifest ลง Drive | EN: Save the manifest to Drive.
print(manifest.groupby("split").size())  # TH: แสดงจำนวนข้อมูลแต่ละชุด | EN: Display split counts.


In [ ]:
HEALTH_DIR = PROJECT_ROOT / "data/health"  # TH: กำหนดโฟลเดอร์ข้อมูลสุขภาพ | EN: Define the health-data folder.
DEMO_CSV = HEALTH_DIR / "health_features_demo_SYNTHETIC.csv"  # TH: ตั้งชื่อให้เห็นชัดว่าเป็นข้อมูลจำลอง | EN: Name the file clearly as synthetic.
TEMPLATE_CSV = HEALTH_DIR / "health_features_real_TEMPLATE.csv"  # TH: กำหนดไฟล์ต้นแบบสำหรับข้อมูลจริง | EN: Define the real-data template file.
columns = ["sample_id", "site_id", "spatial_block", "state_ndvi", "state_ndmi", "pressure_built_probability", "pressure_vv", "trend_ndvi_slope", "context_precipitation_mm", "context_temperature_c", "MHI"]  # TH: กำหนดคอลัมน์ที่จำเป็น | EN: Define the required columns.
pd.DataFrame(columns=columns).to_csv(TEMPLATE_CSV, index=False)  # TH: สร้างไฟล์ต้นแบบว่าง | EN: Create an empty real-data template.
rng = np.random.default_rng(SEED)  # TH: สร้างตัวสุ่มสำหรับข้อมูลสาธิต | EN: Create a generator for demonstration data.
n_rows = 240  # TH: กำหนดจำนวนแถวข้อมูลจำลอง | EN: Set the synthetic row count.
frame = pd.DataFrame({"sample_id": [f"demo_{i:04d}" for i in range(n_rows)], "site_id": [f"site_{i:03d}" for i in range(n_rows)], "spatial_block": [f"block_{i // 10:02d}" for i in range(n_rows)]})  # TH: สร้างรหัสตัวอย่างและบล็อกเชิงพื้นที่ | EN: Create sample IDs and spatial blocks.
frame["state_ndvi"] = rng.uniform(0.20, 0.90, n_rows)  # TH: จำลอง NDVI ซึ่งเป็นตัวแปรสภาพปัจจุบัน | EN: Simulate NDVI as a state variable.
frame["state_ndmi"] = rng.uniform(0.05, 0.75, n_rows)  # TH: จำลอง NDMI ซึ่งเป็นตัวแปรสภาพปัจจุบัน | EN: Simulate NDMI as a state variable.
frame["pressure_built_probability"] = rng.uniform(0.00, 0.80, n_rows)  # TH: จำลองแรงกดดันจากพื้นที่ก่อสร้าง | EN: Simulate built-area pressure.
frame["pressure_vv"] = rng.normal(-11.0, 2.0, n_rows)  # TH: จำลองค่า Sentinel-1 VV | EN: Simulate Sentinel-1 VV values.
frame["trend_ndvi_slope"] = rng.normal(0.0, 0.02, n_rows)  # TH: จำลองแนวโน้ม NDVI | EN: Simulate the NDVI trend.
frame["context_precipitation_mm"] = rng.uniform(900.0, 2600.0, n_rows)  # TH: จำลองฝนระดับพื้นที่ | EN: Simulate site-level precipitation.
frame["context_temperature_c"] = rng.uniform(25.0, 31.5, n_rows)  # TH: จำลองอุณหภูมิระดับพื้นที่ | EN: Simulate site-level temperature.
noise = rng.normal(0.0, 4.0, n_rows)  # TH: เพิ่มสัญญาณรบกวนให้ข้อมูลสาธิต | EN: Add noise to the demonstration data.
frame["MHI"] = np.clip(20 + 55 * frame["state_ndvi"] + 20 * frame["state_ndmi"] - 35 * frame["pressure_built_probability"] + 180 * frame["trend_ndvi_slope"] + noise, 0, 100)  # TH: สร้าง MHI จำลอง 0–100 เพื่อทดสอบโค้ดเท่านั้น | EN: Create a synthetic 0–100 MHI for code testing only.
frame["data_status"] = "synthetic_demo_not_ground_truth"  # TH: ติดป้ายสถานะเพื่อป้องกันการนำไปอ้างผลจริง | EN: Label the data to prevent scientific misuse.
frame.to_csv(DEMO_CSV, index=False)  # TH: บันทึกข้อมูลสุขภาพจำลอง | EN: Save the synthetic health data.
print("Synthetic demo only:", DEMO_CSV)  # TH: เตือนตำแหน่งข้อมูลจำลอง | EN: Report the synthetic-data path.
